# CRISPRa screen analysis — run it in your browser

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pmtran5884/crispra-screen/blob/main/notebooks/run_in_colab.ipynb)

QC and hypergeometric gene scoring for a px52-amplified CRISPR-activation
screen. Nothing to install on your own machine — this runs on Google's free
servers. Press play on each cell in order, or **Runtime → Run all**. The whole
notebook finishes in about a minute.

The example files are **synthetic** — generated by `tools/make_example_data.py`
to mimic the amplicon structure, with an enrichment planted on a few arbitrary
genes. They exist to exercise the pipeline; swap in your own FASTQs at the end.


## 1. Install


In [ ]:
!git clone --depth 1 https://github.com/pmtran5884/crispra-screen.git
%pip install -q ./crispra-screen
!crispra-screen --version


## 2. QC a library FASTQ

One R1 FASTQ in, full QC out. By default it uses the sgRNA key bundled with
the package, so you only supply the FASTQ.

You get a 3-page PDF report plus CSVs covering: the read-fate breakdown (what
fraction of reads are the intended amplicon vs contamination), forward-primer
usage, reads per sgRNA, how skewed the library is, how evenly the 4 guides per
gene are represented, a sequencing-depth saturation analysis, and a table of
genes with no or only one detected guide.


In [ ]:
!crispra-screen qc \
    --fastq crispra-screen/example_data/example_control_R1.fastq.gz \
    --name example_control --outdir qc_out
import os; print('\nfiles written:'); print('\n'.join(sorted(os.listdir('qc_out'))))


### The QC summary


In [ ]:
import pandas as pd
pd.read_csv('qc_out/qc_read_summary.csv').T


### Genes with no or only one detected guide
These are the genes you cannot score reliably, because the hypergeometric
needs several guides per gene to say anything.


In [ ]:
pd.read_csv('qc_out/qc_missing_guide_genes.csv').head(10)


### The full report
`qc_out/qc_report.pdf` has every figure. Download it from the file browser on
the left, or render the pages inline:


In [ ]:
!pip -q install pdf2image && apt-get -qq install -y poppler-utils > /dev/null
from pdf2image import convert_from_path
for page in convert_from_path('qc_out/qc_report.pdf', dpi=70):
    display(page)


## 3. Score a treated arm against the control

This computes a log2 fold change per sgRNA and then a gene-level
hypergeometric score, the way the Broad GPP tool does it.

- `--mock` is your control/unselected FASTQ (used here).
- `--use-bundled-mock` instead compares against a pre-counted reference library.
- `--method local` (default) needs no network. `--method apron` submits to the
  Broad GPP portal and falls back to the local scorer if it is unreachable.

Outputs are the full per-gene table and a publication-ready vector volcano.


In [ ]:
!crispra-screen score \
    --treatment crispra-screen/example_data/example_treated_R1.fastq.gz \
    --mock crispra-screen/example_data/example_control_R1.fastq.gz \
    --method local --name example_treated --outdir score_out
import os; print('\nfiles written:'); print('\n'.join(sorted(os.listdir('score_out'))))


### The hit list
`ACIN1` and `B4GALT1` are two of the genes the synthetic data planted an
enrichment on, so seeing them at the top means the pipeline worked.


In [ ]:
import pandas as pd
g = pd.read_csv('score_out/gene_scores_example_treated.csv')
g = g[(g.direction == 'enriched') & (~g.is_control)]
g.nsmallest(10, 'p_value')[['gene','n_guides','mean_lfc','p_value','fdr']]


### The volcano


In [ ]:
from IPython.display import Image
Image('score_out/volcano_example_treated.png')


## 4. Run it on your own data

Upload your FASTQ files using the folder icon in the left sidebar (or the cell
below), then change the paths in the commands above:

```
!crispra-screen qc    --fastq MY_LIBRARY_R1.fastq.gz --outdir qc_out
!crispra-screen score --treatment MY_TREATED_R1.fastq.gz \
                      --mock MY_CONTROL_R1.fastq.gz --outdir score_out
```

If your library uses a different sgRNA set, pass your own key with
`--key my_key.csv` (a CSV with `Guide_Seq` and `Gene` columns).

Downloads: use the file browser on the left, or `files.download('path')`.


In [ ]:
from google.colab import files
# uploaded = files.upload()                       # upload your FASTQ(.gz)
# files.download('qc_out/qc_report.pdf')          # download a result
